In [1]:
# S-last — tight por padrão. Troque MODE='boost' se estiver UNDER.
MODE = 'tight'   # 'tight' = corta over | 'boost' = sobe pares subestimados
OUT  = "submission_s_last.parquet"

import duckdb, os, glob, shutil
from google.colab import files, drive

con = duckdb.connect()

# garante VIEW base (2022)
def ensure_base():
    try:
        con.execute("SELECT 1 FROM base LIMIT 1").fetchone()
        return
    except Exception:
        pass
    # tenta achar/parar no /content; senão faz upload
    cands = glob.glob('/content/*.parquet')
    if not cands:
        drive.mount('/content/drive', force_remount=True)
        cands = glob.glob('/content/drive/MyDrive/**/df_merge_clean.parquet', recursive=True)
    if not cands:
        up = files.upload()  # selecione df_merge_clean.parquet
        name = list(up.keys())[0]
        shutil.move(name, "/content/df_merge_clean.parquet")
        cands = ['/content/df_merge_clean.parquet']
    PATH = cands[0]
    con.execute(f"CREATE OR REPLACE VIEW raw AS SELECT * FROM read_parquet('{PATH}')")
    con.execute("""
    CREATE OR REPLACE VIEW base AS
    SELECT premise, categoria, pdv, produto,
           CAST(iso_week AS INT) AS iso_week,
           SUM(quantity) AS qty
    FROM raw
    WHERE iso_year=2022
    GROUP BY 1,2,3,4,5
    """)
ensure_base()

# parâmetros por modo
if MODE=='tight':
    TARGET_SHARE = 0.949   # menos cobertura, reduz cauda
    CAP_Q        = 0.88
    CAP_K        = 1.01
    GAMMA        = 0.97
    ROUND_EXPR   = "FLOOR(quantidade)"
else:  # boost
    TARGET_SHARE = 0.962
    CAP_Q        = 0.95
    CAP_K        = 1.08
    GAMMA        = 1.03
    ROUND_EXPR   = "ROUND(quantidade)"

sql = f"""
WITH recent AS (
  SELECT premise,categoria,pdv,produto,
         SUM(qty) FILTER (WHERE iso_week BETWEEN 45 AND 52) AS qty_recent
  FROM base GROUP BY 1,2,3,4
),
ranked AS (
  SELECT *,
         SUM(qty_recent) OVER (PARTITION BY premise,categoria) AS grp_sum,
         SUM(qty_recent) OVER (PARTITION BY premise,categoria
                               ORDER BY qty_recent DESC
                               ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS csum
  FROM recent
),
cut AS (
  SELECT premise,categoria,pdv,produto
  FROM ranked
  WHERE qty_recent>0 AND csum/NULLIF(grp_sum,0) <= {TARGET_SHARE}
),
weekly AS (
  SELECT b.premise,b.categoria,b.pdv,b.produto,b.iso_week, SUM(qty) qty
  FROM base b JOIN cut c USING(premise,categoria,pdv,produto)
  GROUP BY 1,2,3,4,5
),
wmean AS (
  SELECT premise,categoria,pdv,produto,
         (0.60*AVG(qty) FILTER (WHERE iso_week=52)
        + 0.25*AVG(qty) FILTER (WHERE iso_week=51)
        + 0.10*AVG(qty) FILTER (WHERE iso_week=50)
        + 0.05*AVG(qty) FILTER (WHERE iso_week=49)) AS seed_wdecay,
         AVG(qty) FILTER (WHERE iso_week BETWEEN 49 AND 52) AS seed_mean4
  FROM weekly GROUP BY 1,2,3,4
),
lnz8 AS (
  SELECT premise,categoria,pdv,produto, FIRST(qty) AS seed_lnz8
  FROM (
    SELECT w.*, ROW_NUMBER() OVER (PARTITION BY premise,categoria,pdv,produto
                                   ORDER BY iso_week DESC) rn
    FROM weekly w
    WHERE iso_week BETWEEN 45 AND 52 AND qty>0
  ) t WHERE rn=1
  GROUP BY premise,categoria,pdv,produto
),
season AS (
  SELECT premise,categoria,
         NULLIF(AVG(qty) FILTER (WHERE iso_week BETWEEN 1 AND 5),0)
         / NULLIF(AVG(qty) FILTER (WHERE iso_week BETWEEN 45 AND 52),0) AS s_jan_rel
  FROM base GROUP BY 1,2
),
seed0 AS (
  SELECT w.premise,w.categoria,w.pdv,w.produto,
         0.7*w.seed_wdecay + 0.3*COALESCE(l.seed_lnz8, w.seed_mean4, 0.1) AS qty_seed
  FROM wmean w LEFT JOIN lnz8 l USING(premise,categoria,pdv,produto)
),
caps AS (
  SELECT premise,categoria, quantile_cont(qty_seed,{CAP_Q}) AS pcap
  FROM seed0 GROUP BY 1,2
),
seed_adj AS (
  SELECT s.pdv, s.produto,
         GREATEST(0.0, LEAST(
           {GAMMA} * s.qty_seed * COALESCE(se.s_jan_rel,1.0),
           c.pcap * {CAP_K}
         )) AS quantidade
  FROM seed0 s
  LEFT JOIN season se USING(premise,categoria)
  LEFT JOIN caps   c  USING(premise,categoria)
),
weeks AS (SELECT * FROM (VALUES (1),(2),(3),(4),(5)) t(semana))
SELECT
  CAST(semana AS INT) AS semana,
  CAST(pdv AS BIGINT) AS pdv,
  CAST(produto AS BIGINT) AS produto,
  CAST({ROUND_EXPR} AS BIGINT) AS quantidade
FROM weeks CROSS JOIN seed_adj
"""

sub = con.execute(sql).df()
print("rows:", len(sub), "| dups:", sub.duplicated(['semana','pdv','produto']).sum())
assert len(sub) <= 1_500_000, "Passe o limite. Reduza TARGET_SHARE."
con.execute(f"COPY ({sql}) TO '{OUT}' (FORMAT 'parquet')")
from google.colab import files; files.download(OUT)

Mounted at /content/drive


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

rows: 1457250 | dups: 0


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>